# [실습 2] AutoModelForCausalLM 직접 추론과 출력 해석


## 실습 개요

> "Pipeline 없이 작은 LLM의 입력과 출력을 직접 다루면 무엇을 더 이해할 수 있을까?"

이번 실습에서는 `HuggingFaceTB/SmolLM2-135M-Instruct`을 `AutoTokenizer`, `AutoConfig`, `AutoModel`, `AutoModelForCausalLM`로 직접 로드합니다.  
Pipeline 내부에서 자동으로 처리하던 토크나이징, forward, logits 해석, hidden state 요약 과정을 직접 실행합니다.

분류 모델의 logits가 클래스 점수라면, Causal LM의 logits는 각 위치에서 다음 토큰 후보 전체에 대한 점수입니다.


## 실습 목표

1. Auto 클래스가 checkpoint 이름으로 적절한 객체를 로드하는 방식을 이해한다.
2. 토크나이저 출력(`input_ids`, `attention_mask`)을 직접 확인한다.
3. Causal LM의 `logits` shape과 다음 토큰 확률을 해석한다.
4. `AutoModel`의 `last_hidden_state`를 문장 표현으로 요약한다.
5. [TODO] 다음 토큰 Top-k 후보 추출 함수를 작성한다.


## 데이터 출처

- 데이터셋: 실습용 문장 직접 구성
- 모델: `HuggingFaceTB/SmolLM2-135M-Instruct` (Hugging Face Hub)


## 목차

1. 라이브러리 임포트 및 모델 로드
2. 토크나이저 출력 구조 확인
3. Causal LM logits 해석
4. 다음 토큰 확률 Top-k 확인
5. 기본 모델의 hidden state 활용
6. [TODO] 다음 토큰 후보 추출 함수 만들기


---
## 1. 라이브러리 임포트 및 모델 로드


In [1]:
import warnings
warnings.filterwarnings('ignore')

import torch
import pandas as pd
from transformers import AutoConfig, AutoTokenizer, AutoModel, AutoModelForCausalLM

checkpoint = 'HuggingFaceTB/SmolLM2-135M-Instruct'

config = AutoConfig.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
lm_model = AutoModelForCausalLM.from_pretrained(checkpoint)
base_model = AutoModel.from_pretrained(checkpoint)

lm_model.eval()
base_model.eval()

print(f'model_type: {config.model_type}')
print(f'vocab_size: {config.vocab_size}')
print(f'tokenizer: {type(tokenizer).__name__}')

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 15333.40it/s]

model_type: llama
vocab_size: 49152
tokenizer: GPT2Tokenizer


---
## 2. 토크나이저 출력 구조 확인

모델에 텍스트를 바로 넣을 수는 없습니다.  
토크나이저는 텍스트를 `input_ids`, `attention_mask` 등 텐서 형태로 변환합니다.


In [2]:
texts = [
    'Transformers hide many details behind a simple API.',
    'Direct inference helps us understand model outputs.',
]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors='pt')

for key, value in inputs.items():
    print(f'{key:<15} shape={tuple(value.shape)}')

print('\n첫 번째 문장의 input_ids:')
print(inputs['input_ids'][0].tolist())
print('\n디코딩 결과:')
print(tokenizer.decode(inputs['input_ids'][0]))

input_ids       shape=(2, 10)
attention_mask  shape=(2, 10)

첫 번째 문장의 input_ids:
[31806, 366, 11161, 800, 3841, 2893, 253, 2232, 12077, 30]

디코딩 결과:
Transformers hide many details behind a simple API.


---
## 3. Causal LM logits 해석

`AutoModelForCausalLM` 출력의 `logits`는 각 위치에서 다음 토큰 후보 전체에 대한 점수입니다.  
shape은 `(batch_size, seq_len, vocab_size)`입니다.


In [3]:
with torch.no_grad():
    lm_outputs = lm_model(**inputs)

logits = lm_outputs.logits
print('logits shape:', tuple(logits.shape))

last_token_logits = logits[:, -1, :]
print('마지막 위치 logits shape:', tuple(last_token_logits.shape))

logits shape: (2, 10, 49152)
마지막 위치 logits shape: (2, 49152)


---
## 4. 다음 토큰 확률 Top-k 확인

마지막 위치의 logits에 softmax를 적용하면 다음 토큰 확률 분포를 얻을 수 있습니다.  
`torch.topk`로 가장 가능성이 높은 후보 토큰을 확인합니다.


In [ ]:
probs = torch.softmax(last_token_logits, dim=-1)
top_probs, top_ids = torch.topk(probs, k=5, dim=-1)

for row_idx, text in enumerate(texts):
    print(f'입력: {text}')
    for token_id, prob in zip(top_ids[row_idx], top_probs[row_idx]):
        # 토큰 ID를 실제 토큰 문자열로 디코딩
        token = tokenizer.decode([token_id.item()])
        print(f'  {token!r:<12} {prob.item():.6f}')
    print()

입력: Transformers hide many details behind a simple API.
  '\n'         0.182617
  ' This'      0.046143
  ' The'       0.046143
  ' They'      0.035889
  ' If'        0.031738

입력: Direct inference helps us understand model outputs.
  '\n'         1.000000
  '@'          0.000553
  '\n\n'       0.000488
  'box'        0.000096
  'aid'        0.000028



---
## 5. 기본 모델의 hidden state 활용

`AutoModel`은 특정 생성 헤드 없이 기본 모델만 로드합니다.  
출력의 `last_hidden_state`는 각 토큰의 문맥 임베딩이며, attention mask로 실제 토큰만 평균내 문장 표현을 만들 수 있습니다.


In [14]:
with torch.no_grad():
    base_outputs = base_model(**inputs)

hidden = base_outputs.last_hidden_state
# attention_mask을 활용하여 패딩 토큰의 영향을 제거한 평균 풀링 수행

print(inputs['attention_mask'].shape)
mask = inputs['attention_mask'].unsqueeze(2)
print(tuple(mask.shape))

masked_hidden = hidden * mask
sentence_embeddings = masked_hidden.sum(dim=1) / mask.sum(dim=1)

print('last_hidden_state shape:', tuple(hidden.shape))
print('sentence_embeddings shape:', tuple(sentence_embeddings.shape))

torch.Size([2, 10])
(2, 10, 1)
last_hidden_state shape: (2, 10, 576)
sentence_embeddings shape: (2, 576)


---
### [TODO] 다음 토큰 후보 추출 함수 만들기

입력 문장을 토크나이징한 뒤, Causal LM이 예측한 다음 토큰의 Top-k 후보를 DataFrame으로 반환합니다.

`lm_model(**model_inputs)`로 바로 실행하려면 토크나이저 결과가 PyTorch 텐서여야 합니다. `tokenizer()` 호출의 `return_tensors` 값을 `'pt'`로 지정하세요. 이후의 logits 추출, softmax, top-k 계산, DataFrame 구성 코드는 그대로 사용합니다.


In [15]:
def next_token_topk(text, k=5):
    model_inputs = tokenizer(
        text,
        return_tensors='pt',  # TODO: PyTorch 텐서로 반환되도록 수정
    )

    with torch.no_grad():
        logits = lm_model(**model_inputs).logits[:, -1, :]

    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k=k, dim=-1)

    result = pd.DataFrame({
        'token': [tokenizer.decode([token_id]) for token_id in top_ids[0]],
        'token_id': top_ids[0].tolist(),
        'probability': top_probs[0].tolist(),
    })
    return result


In [16]:
topk_result = next_token_topk('The purpose of AutoModel is', k=5)
print(topk_result)
print('행 개수:', len(topk_result))
print('확률 범위:', topk_result['probability'].min(), '~', topk_result['probability'].max())

   token  token_id  probability
0     to       288     0.910156
1    not       441     0.011475
2   that       338     0.008911
3      :        42     0.008911
4    the       260     0.003281
행 개수: 5
확률 범위: 0.0032806396484375 ~ 0.91015625
